# LR/HR 쌍을 어떻게 만들 것인가

지금까지는 데이터를 주어진 것으로 두고 모델만 바꿨다. 여기서는 반대로 한다 —
**모델(EDSR)을 고정하고 학습 쌍을 만드는 방법만 바꾼다.**

| | 전략 | LR 을 어떻게 얻나 | 장점 | 약점 |
|---|---|---|---|---|
| ① | 합성 열화 | HR 을 bicubic 으로 1/3 축소 | 정합이 완벽하다 | 실제 센서 특성이 없다 |
| ② | 실측 페어 | 실제 Sentinel-2 촬영본 | 센서 특성이 진짜다 | 기하·방사·시간차가 남는다 |
| ③ | 고차 열화 | HR 에 블러·노이즈·JPEG 를 무작위로 2회 | 실제와 비슷한 분포 | 열화가 매번 달라 학습이 오래 걸린다 |
| ④ | 기존 g_LR | 이 프로젝트가 원래 쓰던 합성 LR | — | — |

**HR 타일은 네 전략이 완전히 같다. LR 만 다르다.**
검증도 넷 다 **실제 Sentinel-2 LR** 로 통일했다 — 최종 목표가 실제 영상에 쓰는 것이므로.

## 1. 준비

In [ ]:
import sys, json, urllib.request

LIB = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main/lib'
for m in ['sr_utils.py', 'pair_degrade.py']:
    urllib.request.urlretrieve(f'{LIB}/{m}', m)
    sys.modules.pop(m[:-3], None)
from sr_utils import *

RES = f'{BASE}/results/pairs'
V = 'p1'                       # 내려받은 파일 캐시 폴더
META = json.load(open(fetch(f'{RES}/meta.json', f'{V}/meta.json')))
NAMES = ['1_synthetic', '2_real_paired', '3_high_order', '4_gLR']
LBL = {'1_synthetic': '(1) synthetic', '2_real_paired': '(2) real paired',
       '3_high_order': '(3) high-order', '4_gLR': '(4) g_LR'}

def load(scene, name):
    return imageio.imread(fetch(f'{RES}/{scene}/{name}.png', f'{V}/{scene}/{name}.png'))

print('준비 완료 —', ', '.join(META))

## 2. 열화를 직접 만들어 본다

①과 ③은 HR 만 있으면 코드로 만들 수 있다. 저장소에서 HR 을 받아 돌려 보자.
②는 만들 수 없다 — 실제로 촬영해야 얻는다.

In [ ]:
import numpy as np
from pair_degrade import degrade_syn, degrade_high

hr = load('train1', 'HR')
rng = np.random.default_rng(0)

made = {'(1) synthetic': degrade_syn(hr),
        '(3) high-order (seed 0)': degrade_high(hr, np.random.default_rng(0)),
        '(3) high-order (seed 1)': degrade_high(hr, np.random.default_rng(1)),
        '(3) high-order (seed 2)': degrade_high(hr, np.random.default_rng(2))}

fig, ax = plt.subplots(1, 5, figsize=(16, 3.6))
ax[0].imshow(hr); ax[0].set_title(f'HR  {hr.shape[0]}px', fontsize=10)
for a, (n, im) in zip(ax[1:], made.items()):
    a.imshow(im, interpolation='nearest'); a.set_title(f'{n}\n{im.shape[0]}px', fontsize=9)
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

print('고차 열화는 seed 마다 결과가 다르다 — 그것이 목적이다.')

## 3. 네 전략의 LR 비교

같은 HR 에서 나온 (또는 같은 자리를 찍은) 네 가지 LR 이다.

In [ ]:
def sharpness(a):
    return float(cv2.Laplacian(cv2.cvtColor(a, cv2.COLOR_RGB2GRAY), cv2.CV_64F).std())

for s in ('train1', 'train2'):
    hr = load(s, 'HR')
    fig, ax = plt.subplots(1, 5, figsize=(16, 3.6))
    ax[0].imshow(hr); ax[0].set_title('HR (shared)', fontsize=10)
    for a, n in zip(ax[1:], NAMES):
        im = load(s, n)
        a.imshow(im, interpolation='nearest')
        a.set_title(f'{LBL[n]}\nsharpness {sharpness(im):.0f}', fontsize=9)
    for a in ax: a.set_xticks([]); a.set_yticks([])
    fig.suptitle(f"{s} — {META[s]['scene']}", fontsize=11)
    plt.tight_layout(); plt.show()

## 4. 학습 로그

네 전략을 **완전히 같은 조건**으로 5 epoch 씩 학습했다 (EDSR, 처음부터, 486 타일).
검증은 넷 다 실제 Sentinel-2 LR 141 타일이다.

In [ ]:
import pandas as pd

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
COL = dict(zip(NAMES, ['#2f6f9f', '#c96a5b', '#d9a441', '#4f9d69']))
for n, k in zip(NAMES, ['syn', 'real', 'high', 'glr']):
    e = pd.read_csv(fetch(f'{RES}/curve_{k}.csv', f'{V}/curve_{k}.csv'), index_col=0)
    ax[0].plot(e.index, e.l1, marker='o', ms=4, color=COL[n], label=LBL[n])
    ax[1].plot(e.index, e.PSNR, marker='o', ms=4, color=COL[n], label=LBL[n])
ax[0].set_title('Training L1 loss'); ax[0].set_ylabel('L1')
ax[1].set_title('Validation PSNR (real Sentinel-2 LR)'); ax[1].set_ylabel('dB')
for a in ax: a.set_xlabel('epoch'); a.grid(alpha=.3); a.legend(fontsize=8)
plt.tight_layout(); plt.show()

print('학습 손실이 낮다고 검증이 좋은 것이 아니다 — 축이 서로 다르다.')

## 5. 복원 결과

같은 실제 Sentinel-2 입력을 네 모델에 넣은 결과다.

In [ ]:
for s in ('val1', 'val2'):
    panels = [('Original LR', nearest(load(s, 'LR'))), ('Bicubic', load(s, 'Bicubic'))]
    panels += [(LBL[n], load(s, n)) for n in NAMES]
    panels += [('Target HR', load(s, 'HR'))]
    zoom(panels, size=55, title=f"{s} - {META[s]['scene']} (real Sentinel-2 input)")

## 6. 정량 비교

검증 141 타일 전체 평균이다. 입력은 모두 같은 실제 Sentinel-2 LR 이다.

In [ ]:
df = pd.read_csv(fetch(f'{RES}/metrics.csv', f'{V}/metrics.csv')).set_index('strategy')
print(df.to_string(float_format=lambda v: f'{v:.4f}'))

m = df[df.epochs == 5]
base = df.loc['Bicubic']
idx = np.arange(len(m))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, col in zip(ax, ['PSNR', 'SSIM']):
    a.bar(idx, m[col], 0.6, color=[COL[n] for n in m.index])
    a.axhline(base[col], color='#888', ls='--', lw=1.4, label='Bicubic')
    a.set_xticks(idx); a.set_xticklabels([LBL[n] for n in m.index], rotation=20, fontsize=9)
    a.set_title(f'{col} — 5 epoch, real Sentinel-2 val')
    a.grid(axis='y', alpha=.3); a.legend(fontsize=8)
    lo, hi = min(m[col].min(), base[col]), max(m[col].max(), base[col])
    a.set_ylim(lo - (hi - lo) * .4, hi + (hi - lo) * .2)
plt.tight_layout(); plt.show()

## 7. 정리

**② 실측 페어가 이긴다.** PSNR 15.73 으로 ① 합성(15.26)보다 0.47 dB, ③ 고차(14.91)보다
0.81 dB 높다. 학습 곡선에서도 ② 만 5 epoch 내내 단조 상승한다.

**왜 ① 이 지는가.** bicubic 축소본은 실제 센서보다 2배 선명하다(sharpness 110 대 52).
실제 Sentinel-2 가 담지 못하는 디테일이 LR 에 남아 있어, 모델이 "이만큼 선명한 입력"을
전제로 학습된다. 진짜 흐린 영상을 만나면 그 전제가 깨진다.

**③ 이 꼴찌인 것은 조건 탓이 크다.** 통계는 실제에 잘 맞췄지만(sharpness 58 대 52),
열화가 타일마다 무작위라 분포가 넓다. 5 epoch 으로는 그 분포를 다 배우지 못한다.
Real-ESRGAN 이 수십만 iteration 을 쓰는 이유다. **이 표만 보고 ③ 이 나쁜 방법이라고
결론지으면 안 된다.**

**SSIM 은 넷 다 bicubic 보다 낮다.** 5 epoch 은 구조를 살리기에 너무 짧다.
표의 `ref_deployed_gLR`(30 epoch + 이전 학습 계보)은 SSIM 0.4506 으로 유일하게
bicubic 을 넘는다 — 학습량이 다르므로 동등 비교가 아니라 참고선이다.